In [2]:
# Libraries
# ---------------------------------------------------------
import os              # For interacting with the operating system (paths, directories, etc.)
import re              # For regular expressions and text pattern matching
import math            # For basic and advanced mathematical operations
from datetime import datetime, timedelta  # For date and time manipulation
from collections import defaultdict, Counter  # For counting and organizing data efficiently
from pathlib import Path  # For object-oriented filesystem path operations

# ---------------------------------------------------------
# Scientific Libraries
# ---------------------------------------------------------
import numpy as np      # For efficient numerical computations and array operations
import pandas as pd     # For structured data manipulation and analysis
from scipy.stats import mode  # For statistical computations (e.g., mode calculation)

# ---------------------------------------------------------
# Machine Learning Libraries
# ---------------------------------------------------------
import torch                        # Core PyTorch library for tensor operations
import torch.nn as nn               # Neural network components (layers, loss functions, etc.)
import torch.optim as optim         # Optimizers for training neural networks
from torch.utils.data import Dataset, DataLoader, random_split  # Dataset and batching utilities
from sklearn.metrics import classification_report  # For evaluating classification model performance

# ---------------------------------------------------------
# Domain-Specific and Utility Libraries
# ---------------------------------------------------------
import netCDF4 as nc   # For handling NetCDF data formats (common in environmental and geospatial data)
import psutil          # For system and process monitoring (useful for memory and CPU profiling)       


In [10]:
# --- CNN model --- 

class PhaseCNN(nn.Module):
    """
    A hybrid CNN–BiLSTM–Transformer model for temporal pattern recognition.
    
    Parameters
    ----------
    window_size : int
        Length of the input time window (sequence length).
    
    Architecture Details
    --------------------
    - Four CNN branches with kernel sizes {3, 5, 7, 11}
      capture multi-scale temporal features.
    - Each branch has four Conv1D layers with increasing channels.
    - Outputs are concatenated and fed into a BiLSTM for sequential modeling.
    - A Transformer encoder refines temporal dependencies.
    - A fully connected layer performs binary classification.
    """    
    def __init__(self, window_size):
        super().__init__()

        # ---- Helper Function: CNN Branch Constructor ---- #
        def make_branch(kernel_size):
            return nn.Sequential(
                nn.Conv1d(1, 16, kernel_size=kernel_size, padding=kernel_size // 2),
                nn.ReLU(),
                
                nn.Conv1d(16, 32, kernel_size=kernel_size, padding=kernel_size // 2),
                nn.ReLU(),
                
                nn.Conv1d(32, 64, kernel_size=kernel_size, padding=kernel_size // 2),
                nn.ReLU(),
        
                nn.Conv1d(64, 128, kernel_size=kernel_size, padding=kernel_size // 2),
                nn.ReLU(),
        
            )
        
         # ---- CNN Branches with Different Kernel Sizes ---- #
        self.branch1 = make_branch(kernel_size=3)
        self.branch2 = make_branch(kernel_size=5)
        self.branch3 = make_branch(kernel_size=7)
        self.branch4 = make_branch(kernel_size=11)
        
        # ---- BiLSTM for Temporal Modeling ---- # 
        self.lstm = nn.LSTM(
            input_size=128 * 4,       # Four concatenated feature maps
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )
        
        # ---- Transformer Encoder for Contextual Attention ---- #
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=512,      # from 2 * BiLSTM
            nhead=8,          # must satisfy:  d_model % nhead == 0
            dim_feedforward=512 * 2,    # Expansion factor for MLP inside Transformer
            dropout=0.1,
            batch_first=True  
        )
        
        # ---- Classification Head ---- #
        self.pos_encoder = PositionalEncoding(d_model=512)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(256 * 2, 2)  # binary classification (rise vs. else)
        
        def forward(self, x):
            """
            Defines the forward pass through the network.
            
            Args:
                x: Input tensor of shape (batch_size, 1, sequence_length)
            Returns:
                out: Logits of shape (batch_size, sequence_length, 2)
            """
            
            # ---- Feature Extraction from Multiple CNN Branches ---- #
            b1 = self.branch1(x)  
            b2 = self.branch2(x)  
            b3 = self.branch3(x)
            b4 = self.branch4(x)
            
            # Concatenate feature maps across the channel dimension
            x = torch.cat([b1, b2, b3, b4], dim=1)   # Shape: (B, 512, T)
            
            # Prepare for LSTM: (B, T, C)
            x = x.permute(0, 2, 1)
            
             # ---- Sequential Modeling ---- #
            lstm_out, _ = self.lstm(x)    # (B, T, 512)
            x = self.dropout(lstm_out)
            
            # ---- Contextual Encoding via Transformer ---- #
            x = self.pos_encoder(x)
            x = self.transformer(x)
            
            # ---- Final Classification ---- #
            out = self.classifier(x)     # (B, T, 2)
            return out


class TimestampDataset(torch.utils.data.Dataset):

    """
    Custom Dataset for time series data with corresponding labels and timestamps.

    Parameters
    ----------
    X : np.ndarray or torch.Tensor
        Input features of shape (N, W), where W is the window size.
    y : np.ndarray or torch.Tensor
        Target labels corresponding to each window.
    timestamps : list-like
        List of timestamps associated with each data sample.
    """
    
    def __init__(self, X, y, timestamps):
        # Ensure tensor format and channel dimension (N, 1, W)
        if isinstance(X, torch.Tensor):
            self.X = X.clone().detach().unsqueeze(1)  
        else:
            self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)

        # Labels
        if isinstance(y, torch.Tensor):
            self.y = y.clone().detach()
        else:
            self.y = torch.tensor(y, dtype=torch.long)     # (N, W)

        # Convert timestamps to string format for consistency            
        self.ts = [str(ts) for ts in timestamps]

    def __len__(self):
        """
        Returns total number of samples.
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        Returns a single sample consisting of:
        - Feature tensor (1 × W)
        - Label
        - Timestamp (as string)
        """
        return self.X[idx], self.y[idx], self.ts[idx]


class PositionalEncoding(nn.Module):
    """
    Implements sinusoidal positional encoding as described in:
    "Attention Is All You Need" (Vaswani et al., 2017).

    Adds position-dependent information to sequential data,
    allowing the Transformer to recognize order without recurrence.
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # Create positional encodings of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        # Apply sine and cosine functions to even and odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension (1, max_len, d_model)
        pe = pe.unsqueeze(0)  
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Adds positional encodings to the input tensor.
        Args:
            x: Tensor of shape (batch_size, sequence_length, d_model)
        Returns:
            Tensor with positional encodings added.
        """
        # Scale the input to preserve variance across layers
        x = x * torch.sqrt(torch.tensor(x.size(-1), dtype=torch.float32, device=x.device))
        # Add precomputed positional encodings
        x = x + self.pe[:, :x.size(1), :]
        return x

In [11]:
# --- Training Script ---
def train_cnn_on_date_range(data_train, data_test, search_seconds, stride_fraction, n_epoch):

    """
    Train the PhaseCNN model over a specified date range.

    Parameters
    ----------
    data_train : dict
        Dictionary containing training data with keys:
            - "X": Feature array (list or np.ndarray)
            - "y": Label array
            - "timestamps": Corresponding timestamps
    data_test : dict
        Dictionary with the same structure as `data_train` for testing.
    search_seconds : int
        Input window size for the CNN model.
    stride_fraction : float
        (Unused parameter, retained for compatibility or future work.)
    n_epoch : int
        Number of training epochs.

    Returns
    -------
    model : PhaseCNN
        Trained model instance.
    report_df : pandas.DataFrame
        Classification report as a DataFrame (precision, recall, f1-score, etc.).
    all_labels : list
        Ground truth labels for the test set.
    all_preds : list
        Predicted labels for the test set.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    torch.autograd.set_detect_anomaly(True)  # Enable autograd anomaly detection

    # samples for training
    X_train = data_train["X"]
    y_train = data_train["y"]
    
    # samples for evaluation
    X_test = data_test["X"]
    y_test = data_test["y"]
    
    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError("Training or testing data is empty. Check input dates and files.")

    # Compute Class Weights (for Imbalanced Data)
    counts = Counter(np.array(y_train).flatten())
    total = sum(counts.values())

    # Inverse-frequency weighting
    class_weights = [total / counts[i] if counts[i] > 0 else 0.0 for i in range(2)]
    class_weights = torch.tensor(class_weights, dtype=torch.float32)
    class_weights = class_weights / class_weights.sum() # Normalize
    class_weights = class_weights.to(device)

    # data sets
    train_dataset = TimestampDataset(np.array(X_train), np.array(y_train), data_train["timestamps"])
    test_dataset  = TimestampDataset(np.array(X_test),  np.array(y_test),  data_test["timestamps"])
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16)

    # GPU
    model = PhaseCNN(search_seconds).to(device)
    # model = torch.compile(model) # on linux
    
    optimizer = optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.CrossEntropyLoss(weight=class_weights) 

    # Training Loop
    for epoch in range(n_epoch):
        model.train()
        total_loss = 0
        
        for batch_idx, (xb, yb, _) in enumerate(train_loader):
            # Time estimation for the first batch
            if batch_idx == 0:
                start_time = time.time()
                
            # print(f"Processing batch of shape: {xb.shape}")
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()   # Reset for next batch
            
            out = model(xb)   # Forward pass
            loss = criterion(out.view(-1, 2), yb.view(-1))  # (B*L, 2), (B*L)       

            # Print RAM usage
            # print(f"[Batch] RAM used: {psutil.virtual_memory().percent:.2f}%")
            # print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

            # Backpropagation and parameter update
            loss.backward() # Gradients (w.r.t. weights)
            optimizer.step() 
            total_loss += loss.item()

            # Estimate epoch duration based on first batch
            if batch_idx == 0:
                end_time = time.time()
                first_batch_time = end_time - start_time
                print(f"Estimated time per batch: {first_batch_time:.2f} sec")
                
                steps_per_epoch = len(train_loader)
                est_epoch_time = steps_per_epoch * first_batch_time
                print(f"Estimated epoch time: {est_epoch_time:.2f} sec ≈ {est_epoch_time/60:.2f} min")
        
        # Average loss per sample for interpretability
        # print(f"Epoch {epoch} - Loss: {total_loss:.4f}")
        total_samples = len(train_loader.dataset)
        avg_loss = total_loss / total_samples        
            
        print(f"Epoch {epoch} - Avg Loss per sample: {avg_loss:.4f}")

    # Evaluation Loop
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for xb, yb, _ in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            out = model(xb)                  # (B, L, 2)
            preds = out.argmax(dim=2)        # (B, L)
            all_preds.extend(preds.view(-1).cpu().numpy())
            all_labels.extend(yb.view(-1).cpu().numpy())

    report = classification_report(all_labels, all_preds, target_names=["background", "event"], output_dict=True, zero_division=0)
    return model, pd.DataFrame(report).T , all_labels, all_preds


In [12]:
# Post-process daily predictions into events 
def extract_events_from_timeline(voted_preds):

    """
    Extracts discrete events from a dictionary of timestamped predictions.

    Parameters
    ----------
    voted_preds : dict
        Dictionary mapping timestamps to predicted labels.
        Format: {timestamp: int}, where label ∈ {0: background, 1: rise phase}.

    Returns
    -------
    events : list of dict
        Each event is represented as:
        {
            'start': timestamp of event onset,
            'peak' : timestamp of last detected event frame
        }

    """
    events = []         # List to store all identified event intervals
    curr_event = None   # Temporary holder for the currently active event

    # Iterate through predictions
    for t in sorted(voted_preds.keys()):
        label = voted_preds[t]

        # --- Case 1: Rise/event ---
        if label == 1:  
            # Start a new event if none is active
            if curr_event is None:
                curr_event = {'start': t, 'peak': t}
            else:
                # Extend the event duration to the current timestamp
                if t > curr_event['peak']:
                    curr_event['peak'] = t
        
        # --- Case 2: Background ---
        else:  
            if curr_event:
                events.append(curr_event)
                curr_event = None

    # Append the last active event if it continued until the end
    if curr_event:
        events.append(curr_event)

    return events



In [ ]:
import time
start_time = time.time()

# Load Training and Testing Data
data_train = np.load("Train_set_manual.npz")
data_test = np.load("Test_set_manual.npz")

# Datasets are expected in NumPy `.npz` format containing:
#   - 'X' : input features
#   - 'y' : target labels
#   - 'timestamps' : corresponding time metadata

search_seconds = 600     # Window size in seconds (temporal input span)
stride_fraction = 0.2    # Fractional overlap between input windows
n_epoch = 10             # Number of full training iterations

# Train the Model
model, report_df, all_labels, all_preds = train_cnn_on_date_range(
    data_train = data_train, 
    data_test = data_test,     
    search_seconds = search_seconds,
    stride_fraction = stride_fraction,
    n_epoch = n_epoch
)

# Post-training
model.eval()
model.cuda()

# Save state_dict
torch.save(model.state_dict(), "CNN_Model_Manual_4layer_BiLSTM_Trans_Encoder.pt")

# Execution Time and Report Summary
end_time = time.time()
elapsed = end_time - start_time

print(f"Execution time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
print("=== Classification Report ===")
display(report_df)

In [13]:
# Load model 
search_seconds = 600
stride_fraction = 0.2
model = PhaseCNN(window_size = search_seconds)
model.load_state_dict(torch.load("CNN_Model_Manual_4layer_BiLSTM_Trans_Encoder.pt", weights_only=True, map_location=torch.device("cpu")))
model.eval()
model.cuda()

PhaseCNN(
  (branch1): Sequential(
    (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
    (4): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): ReLU()
    (6): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): ReLU()
  )
  (branch2): Sequential(
    (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (3): ReLU()
    (4): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): ReLU()
    (6): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (7): ReLU()
  )
  (branch3): Sequential(
    (0): Conv1d(1, 16, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): Conv1d(16, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (3): ReLU()
    (4): Conv1d(32, 64, kernel_size=(7,), stride=(1,), pad

In [14]:
# --- Process each year ---
import time
for year in range(2018, 2026):
    print(f"\nProcessing year: {year}")
    start_time = time.time()

    set_dir = Path.cwd()
    cat_dir = Path.cwd()
    npz_path = set_dir / f"Prediction_set_{year}.npz"

    # Skip if dataset is missing
    if not Path(npz_path).exists():
        print(f"File not found: {npz_path}, skipping.")
        continue

    # Load prediction set
    npz = np.load(npz_path, allow_pickle=True)
    X = npz['X']
    # y = npz['y']
    # -----------------------------------------------------
    # Each .npz file should contain:
    #   - X : model input tensors
    #   - y : (optional) labels, not required during inference
    #   - timestamps : sequence timestamps
    # -----------------------------------------------------
    

    # Create dummy labels of correct shape (not used during inference, but satisfies the expected __getitem__ output: (X[i], y[i], ts[i]))
    y = np.zeros((X.shape[0], X.shape[1]), dtype=np.uint8)

    # Convert timestamps to string representations
    timestamps = [str(pd.to_datetime(ts)) for ts in npz['timestamps']]

    # Create Dataset and DataLoader
    dataset = TimestampDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long), timestamps)
    dataloader = DataLoader(dataset, batch_size=8, shuffle = False)

    # Model Inference and Daily Aggregation
    predicted_per_day = defaultdict(list)

    with torch.no_grad():
        for xb, yb, tstamp_b in dataloader:
            xb = xb.cuda()
            out = model(xb)  # (B, L 2)
            preds = out.argmax(dim=2).cpu().numpy()   # Predicted labels per timestep

             # Map predictions to timestamps
            for i in range(len(preds)):
                ts = pd.to_datetime(tstamp_b[i])
                for j, label in enumerate(preds[i]):
                    day = ts + timedelta(seconds=j) 
                    predicted_per_day[day].append(label)

    # Majority voting
    # Aggregates multiple overlapping predictions for the same timestamp by selecting the most frequently predicted label.
    voted_preds = {t: Counter(labels).most_common(1)[0][0] for t, labels in predicted_per_day.items()}

    # Extract events
    events = extract_events_from_timeline(voted_preds)

    # Build catalog
    catalog_rows = []
    for ev in events:
        catalog_rows.append({
            'date': ev['start'].date().isoformat(),
            'event_start': ev['start'],
            'event_peak': ev['peak']
        })

    catalog_df = pd.DataFrame(catalog_rows)
    
    # Save Catalog to CSV
    output_path = cat_dir / f"CNN_Catalog_{year}.csv"
    catalog_df.to_csv(output_path, index=False)
    print(f"Saved: {output_path} | Events: {len(catalog_df)}")

    # Report Processing Time
    end_time = time.time()
    print(f"Elapsed time: {end_time - start_time:.2f} sec")



Processing year: 2018
File not found: C:\Users\nfar4944\OneDrive - The University of Sydney (Staff)\Desktop\USYD_Laptop\Projects\Farhang_2024\Reports\The Catalog\Codes\CNN\Prediction_set_2018.npz, skipping.

Processing year: 2019
File not found: C:\Users\nfar4944\OneDrive - The University of Sydney (Staff)\Desktop\USYD_Laptop\Projects\Farhang_2024\Reports\The Catalog\Codes\CNN\Prediction_set_2019.npz, skipping.

Processing year: 2020
File not found: C:\Users\nfar4944\OneDrive - The University of Sydney (Staff)\Desktop\USYD_Laptop\Projects\Farhang_2024\Reports\The Catalog\Codes\CNN\Prediction_set_2020.npz, skipping.

Processing year: 2021
File not found: C:\Users\nfar4944\OneDrive - The University of Sydney (Staff)\Desktop\USYD_Laptop\Projects\Farhang_2024\Reports\The Catalog\Codes\CNN\Prediction_set_2021.npz, skipping.

Processing year: 2022
File not found: C:\Users\nfar4944\OneDrive - The University of Sydney (Staff)\Desktop\USYD_Laptop\Projects\Farhang_2024\Reports\The Catalog\Codes

IndexError: tuple index out of range